In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### Get all programs

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

#--------- chose a disease --------------------
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

In [8]:
verbose=False
force=False

method='deseq2'
imax_tumor=250
imax_normal=50

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

exclude_prog_list=['CCLE']

dfn_tumor, dfn_normal, df_gtex, df_ana = gdc.get_all_data_from_disease(disease_id=DISEASE_ID, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

df_ana



>> dfn_tumor (60616, 460)
>> dfn_normal (60616, 103)


,prog_id,psi_id,disease_id,primary_site
0,CPTAC,PAAD,PAAD,Pancreas
1,CPTAC,PAAD_GDC,PAAD,Pancreas
2,TCGA,TCGA-PAAD,PAAD,Pancreas


In [6]:
fname = 'log_exp_combat_tumor.tsv'
dfn_combat_tumor_final = pdreadcsv(fname, gdc.root_mprog_lfc, verbose=True)

dfn_combat_tumor_final.head(3)


Table opened ((10000, 460)) at '/home/flavio/uv/perturb_agent/data/multi_progs/lfc/log_exp_combat_tumor.tsv'


,geneid,symbol,biotype,1,2,3,4,5,6,7,...,448,449,450,451,452,453,454,455,456,457
0,ENSG00000270641,TSIX,lncRNA,11.203,9.576,11.078,10.160,10.666,6.223,9.880,...,10.190,1.500,10.269,1.728,1.717,10.269,6.737,1.737,9.364,1.718
1,ENSG00000262619,LINC00621,lncRNA,6.601,6.692,9.819,9.716,12.600,4.964,9.189,...,1.184,1.211,1.859,2.084,2.908,1.113,1.764,1.405,1.560,1.785
2,ENSG00000281383,FP671120.7,lncRNA,2.251,-0.808,8.865,-0.808,9.364,4.366,1.978,...,0.534,0.491,0.419,0.528,1.181,0.588,0.282,0.535,0.282,0.687


In [7]:
fname = 'log_exp_combat_normal.tsv'
dfn_combat_normal_final = pdreadcsv(fname, gdc.root_mprog_lfc, verbose=True)

dfn_combat_normal_final.head(3)

Table opened ((10000, 103)) at '/home/flavio/uv/perturb_agent/data/multi_progs/lfc/log_exp_combat_normal.tsv'


,geneid,symbol,biotype,458,459,460,461,462,463,464,...,548,549,550,551,552,553,554,555,556,557
0,ENSG00000270641,TSIX,lncRNA,14.364,5.153,3.100,2.233,4.173,14.392,2.458,...,4.166,4.319,6.311,3.644,4.885,5.878,14.036,5.771,4.915,14.846
1,ENSG00000262619,LINC00621,lncRNA,2.726,3.669,2.395,1.671,2.055,7.157,0.525,...,2.241,2.542,3.140,1.614,1.739,3.301,3.517,5.127,4.779,3.450
2,ENSG00000281383,FP671120.7,lncRNA,-0.157,-0.157,-0.157,-0.157,-0.157,-0.157,-0.157,...,0.322,0.322,0.322,0.322,0.322,0.322,1.656,0.322,1.071,0.322


In [9]:
group = 'Tumor'

verbose=False
force=False

disease_id=DISEASE_ID
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
min_clusters = 6; max_clusters = 12
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05
perc_min_samples=0.25; top_n=10_000

if group == 'Tumor':
    df = dfn_tumor
    n_clusters = 10
else:
    df = dfn_normal
    n_clusters = 3


df_cluster, df_sel, df_cpm, df_pca, df_umap = gdc.cluster_expression_data_group(df, group=group, n_clusters=n_clusters, 
                                                n_components=n_components, min_clusters=min_clusters, max_clusters=max_clusters,
                                                n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                method_hca=method_hca, hca_criterion=hca_criterion,
                                                LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff,
                                                perc_min_samples=perc_min_samples, top_n=top_n,
                                                force=force, verbose=verbose)

In [11]:
df_hca = gdc.df_hca
df_hca

,sample,cluster
0,1,7
1,2,1
2,3,7
3,4,1
4,5,6
...,...,...
452,453,5
453,454,5
454,455,5
455,456,5


In [45]:
force=False
verbose=True

nclu=10

cols = np.array(df_hca[df_hca.cluster == nclu]['sample'])
print(cols)

col_annots = [0, 1, 2]

all_cols = col_annots + list(cols+2)

dfn_combat_tum = dfn_combat_tumor_final.iloc[:, all_cols].copy()

gene_cols = ["geneid", "symbol", "biotype"]
cols2 = np.arange(len(cols))+1

dfn_combat_tum.columns = gene_cols + list(cols2)
dfn_combat_tum.head(3)

[ 48 111 120 131 262 324 333 344]


,geneid,symbol,biotype,1,2,3,4,5,6,7,8
0,ENSG00000270641,TSIX,lncRNA,12.229,6.453,7.608,2.383,13.196,7.842,8.913,4.070
1,ENSG00000262619,LINC00621,lncRNA,7.905,5.633,-0.322,2.034,8.971,6.787,1.060,3.326
2,ENSG00000281383,FP671120.7,lncRNA,9.629,9.754,-0.129,10.702,10.322,10.451,0.282,11.426


In [46]:
dfn_combat_norm = dfn_combat_normal_final.copy()

cols = list(dfn_combat_norm.columns)

gene_cols = ["geneid", "symbol", "biotype"]
cols2 = np.arange(len(cols)-3)+1

dfn_combat_norm.columns = gene_cols + list(cols2)
dfn_combat_norm.head(3)



,geneid,symbol,biotype,1,2,3,4,5,6,7,...,91,92,93,94,95,96,97,98,99,100
0,ENSG00000270641,TSIX,lncRNA,14.364,5.153,3.100,2.233,4.173,14.392,2.458,...,4.166,4.319,6.311,3.644,4.885,5.878,14.036,5.771,4.915,14.846
1,ENSG00000262619,LINC00621,lncRNA,2.726,3.669,2.395,1.671,2.055,7.157,0.525,...,2.241,2.542,3.140,1.614,1.739,3.301,3.517,5.127,4.779,3.450
2,ENSG00000281383,FP671120.7,lncRNA,-0.157,-0.157,-0.157,-0.157,-0.157,-0.157,-0.157,...,0.322,0.322,0.322,0.322,0.322,0.322,1.656,0.322,1.071,0.322


In [ ]:
df_normal = gdc.cdegs.deduplicate_by_max_reads(dfn_combat_norm)


In [47]:
method='deseq2'
force=True
verbose=True

df_lfc, df_lfc_ori, degs_txt, msg = gdc.calc_degs(
                                        prog_id = PROG_ID, 
                                        psi_id = PSI_ID,
                                        ncluster = nclu,
                                        df_tumor = dfn_combat_tum,
                                        df_normal = dfn_combat_norm, 
                                        df_gtex_ctrl = pd.DataFrame(),
                                        root_lfc= gdc.root_mprog_lfc,
                                        root_src = gdc.root_src,
                                        run_conda = True,
                                        lfc_cutoff = 1.0,
                                        fdr_cutoff = 0.05,
                                        method = method,
                                        force = force,
                                        verbose = verbose,
                                        )

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


/home/flavio/uv/perturb_agent/notebooks/../src/libs/calc_degs_lib.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2["_total_reads"] = df2[count_cols].sum(axis=1)


RuntimeError: R DEG script failed -> /home/flavio/uv/perturb_agent/src/libs/calc_degs.R.
Command: conda run -n renv Rscript /home/flavio/uv/perturb_agent/src/libs/calc_degs.R --counts /tmp/tmpxb61vwpb/counts.tsv --meta /tmp/tmpxb61vwpb/meta.tsv --out /tmp/tmpxb61vwpb/lfc_results.tsv --method deseq2 --manual-dispersion 0.1 --min-total-count 10

STDOUT:


STDERR:
Error in DESeqDataSet(se, design = design, ignoreRank) : 
  some values in assay are negative
Calls: run_deseq2 -> DESeqDataSetFromMatrix -> DESeqDataSet
Execution halted
ERROR conda.cli.main_run:execute(142): `conda run Rscript /home/flavio/uv/perturb_agent/src/libs/calc_degs.R --counts /tmp/tmpxb61vwpb/counts.tsv --meta /tmp/tmpxb61vwpb/meta.tsv --out /tmp/tmpxb61vwpb/lfc_results.tsv --method deseq2 --manual-dispersion 0.1 --min-total-count 10` failed. (See above for error)

In [ ]:
df_lfc